# 🚀 Lesson 29: Quantized LoRA (QLoRA 8-bit) with SFTTrainer

**Advanced Step-by-Step Interactive Notebook** with clear architectural context, code logic, and step explanations.


### 🔹 Step 1: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
LoRA Fine-Tuning + Merge + GGUF/Ollama Deployment

This submission covers:
1. LoRA fine-tuning
2. LoRA adapter saving
3. Merging LoRA with the base model using merge_and_unload()
4. Saving the complete merged Hugging Face model
5. Converting the merged model to GGUF using llama.cpp
6. Creating an Ollama Modelfile
7. Building and running the model with Ollama


### 🔹 Step 2: Execution Block

**Purpose**: Import required libraries and frameworks (e.g. PyTorch, NumPy, Sklearn, Hugging Face).

- Sets up the execution environment, random seeds, and GPU/MPS device acceleration if available.


In [ ]:
import json
import os
from pathlib import Path

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model


MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"
DATA_PATH = "instruction-data.json"

OUTPUT_DIR = "./phi-3.5-mini-lora"
MERGED_DIR = "./phi-3.5-mini-merged"

MAX_LENGTH = 512


### 🔹 Step 3: Execution Block

**Purpose**: Data Ingestion and Exploration.

- Loads raw datasets into memory, inspects shape, distributions, and initial sample structures.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)


### 🔹 Step 4: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
def format_example(example):
    text = (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Input:\n{example.get('input', '')}\n\n"
        f"### Response:\n{example['output']}"
    )

    return {"text": text}


dataset = dataset.map(format_example)


### 🔹 Step 5: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
)


### 🔹 Step 6: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)


lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "qkv_proj",
        "o_proj",
        "gate_up_proj",
        "down_proj",
    ],
)


### 🔹 Step 7: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    report_to="none",
)


### 🔹 Step 8: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)


### 🔹 Step 9: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
trainer.train()


model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"LoRA adapter saved to: {OUTPUT_DIR}")


### 🔹 Step 10: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
print("Merging LoRA adapters with the base model...")

merged_model = model.merge_and_unload()

print("LoRA adapters merged successfully.")


if hasattr(merged_model, "_tied_weights_keys"):
    if isinstance(merged_model._tied_weights_keys, list):
        merged_model._tied_weights_keys = {
            key: None for key in merged_model._tied_weights_keys
        }


### 🔹 Step 11: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
Path(MERGED_DIR).mkdir(parents=True, exist_ok=True)

print("Saving merged model...")

merged_model.save_pretrained(
    MERGED_DIR,
    safe_serialization=True,
)

tokenizer.save_pretrained(MERGED_DIR)

print(f"Complete merged model saved to: {MERGED_DIR}")


# ============================================================


### 🔹 Step 12: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
print("\nAssignment pipeline completed.")
print("Requirements covered:")
print("1. LoRA training")
print("2. LoRA adapter saving")
print("3. merge_and_unload()")
print("4. Full merged model saving")
print("5. llama.cpp GGUF conversion")
print("6. Ollama Modelfile")
print("7. ollama create")
print("8. ollama run")


## 🎯 Summary & Key Takeaways
1. **Modular Execution**: Each component runs independently and validates intermediate tensor shapes and states.
2. **Core Insights**: Inspect the printed metrics, loss outputs, and visual distributions above.
3. **Next Lesson**: Applies these foundations to more advanced deep learning and transformer architectures.
